[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/solutions/02_comparing_models.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Comparing models properly

Allow about 45 minutes. Which recordings change decision between two
models, and is the change worth making? You will calculate reclassification
indices, build a paired uncertainty interval, and separate a description of
moving scores from a priced judgement. Recording labels are real CinC labels;
every model score is simulated.

The action is **send this recording for AF review**, priced as in the core
notebook: $C_{TP}$ = 1, $C_{FP}$ = 1, $C_{FN}$ = 5 and $C_{TN}$ = 0, in illustrative loss units.
Replace each `...` and run the checker below it; if you are stuck for about
10 minutes, use the stuck switch below the function, where there is one, or open the worked copy in `extension/solutions/`.
A later cell that needs an unfinished problem prints `Waiting for ...`: once that
problem is done or switched, click in the waiting cell, choose **Runtime > Run
before**, then run the waiting cell.

In [ ]:
import numpy as np
from IPython.display import display
from clinical_utility import data, comparison
from clinical_utility import extension_checks as check, extension_plots as view

cohort = data.load_arrays("development")
y = cohort["af_present"]
old, new = cohort["random_forest"], cohort["resnet"]
cut = .2

## Start with a change in action

Compare the random forest (old) with the raw ResNet (new) on the same
development recordings, at a cut of 0.2. A reclassification table has the old
action in its rows and the new action in its columns, separately for
recordings with and without AF. The off-diagonal cells are changed decisions.

For an AF recording, moving up into review is useful and moving down is
harmful. For a recording without AF, the directions reverse. Net
reclassification improvement (NRI) summarises these moves in two components,
one for AF recordings and one for the rest, and adds them.

### Problem A: Predict

Should each NRI component divide by the recordings in its own outcome group,
or by the whole cohort? Store `"within_outcome"` or `"whole_cohort"`. Then
consider: would adding the two components encode the default prices?

In [ ]:
denominator = "within_outcome"
_ = check.run("cmp_predict", denominator)

#### The two denominators

Each component is a proportion within its own outcome group:

`event = P(up | AF) - P(down | AF)`

`non_event = P(down | no AF) - P(up | no AF)`

On development, the AF component is (74 up minus 9 down) divided by the
341 AF recordings, giving +0.191. Adding the two components weights each
group's proportion equally, whatever the group sizes and whatever a miss or a
false referral costs. So no, the sum does not encode the prices.

In [ ]:
reclassification = comparison.reclassification_table(old, new, y, [cut])
display(view.reclassification(reclassification["events"], reclassification["non_events"]))

## Movement across the cut

### Problem B: Make

Write `categorical_nri(old, new, y, cut)` returning a dictionary with the keys
`event`, `non_event` and `total`. Build the old and new review masks with
`>=`, compare them, then take the two group means:
`event = P(up | AF) - P(down | AF)` and
`non_event = P(down | no AF) - P(up | no AF)`. Do not divide by the number who
moved: recordings that stayed put belong in the denominators.

In [ ]:
def categorical_nri(old, new, y, cut):
    y = np.asarray(y, dtype=bool)
    old_action, new_action = np.asarray(old) >= cut, np.asarray(new) >= cut
    up = new_action & ~old_action
    down = old_action & ~new_action
    event = up[y].mean() - down[y].mean()
    non_event = down[~y].mean() - up[~y].mean()
    return {"event": event, "non_event": non_event, "total": event + non_event}
_ = check.run("cmp_categorical", categorical_nri)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# categorical_nri = check.use_reference("categorical_nri")

In [ ]:
if check.ready(categorical_nri=categorical_nri):
    print("Categorical NRI:", categorical_nri(old, new, y, cut))

The development components are +0.191 and -0.157, totalling +0.034. A
positive total can sit beside many more unnecessary reviews.

## Movement without categories

Continuous NRI counts any strict rise or fall in score as a move; ties count
neither way. The integrated discrimination improvement (IDI) uses the size of
the change: the mean rise among AF recordings plus the mean fall among the
rest. It equals the change in the gap between the two groups' mean scores.

For a recording without AF whose score rises, continuous NRI penalises the
direction of the move and IDI penalises its size. Neither asks whether the
action changes at the service's preference.

### Problem C: Modify

Adapt your categorical calculation into `movement_indices(old, new, y)`. Return
a dictionary with `continuous` and `idi` entries, each holding `event`,
`non_event` and `total`, as in the starter. Use strict comparisons for
continuous NRI and mean differences for IDI.

In [ ]:
def movement_indices(old, new, y):
    old, new, y = np.asarray(old), np.asarray(new), np.asarray(y, dtype=bool)
    up, down = new > old, new < old
    event = up[y].mean() - down[y].mean()
    non_event = down[~y].mean() - up[~y].mean()
    event_idi = (new[y] - old[y]).mean()
    non_idi = (old[~y] - new[~y]).mean()
    return {"continuous": {"event": event, "non_event": non_event, "total": event + non_event},
            "idi": {"event": event_idi, "non_event": non_idi, "total": event_idi + non_idi}}
_ = check.run("cmp_continuous", movement_indices)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# movement_indices = check.use_reference("movement_indices")

In [ ]:
if check.ready(movement_indices=movement_indices):
    print(movement_indices(old, new, y))

NRI can reward miscalibration: raising the scores of enough AF recordings
improves a movement index even when the decisions get worse. Continuous NRI
counts arbitrarily small changes, and IDI depends on the probability scale.
Here the positive NRI favours the raw ResNet despite its lower net benefit.
Report reclassification as a description; judge the policy by net benefit at
a stated preference.

## Put a preference and uncertainty on the comparison

At threshold probability $p_t$, net benefit is
`TP/n - FP/n * p_t/(1 - p_t)`. Each recording contributes its detection value,
its false-review penalty or nothing. A change of model contributes **new minus
old** value, and the difference in net benefit is the mean of those
contributions over all recordings. At $p_t$ = 0.2 a false review weighs
0.25:

| A recording's change of decision | Contribution, new minus old |
|---|---:|
| AF, moves up into review | +1 |
| AF, moves down out of review | -1 |
| No AF, moves up into review | -0.25 |
| No AF, moves down out of review | +0.25 |
| No change | 0 |

**Worked example.** Take five recordings. One with AF moves up, two without AF
move up and two do not change: one with AF stays in review and one without AF
stays out. The contributions sum to 0.50, and their mean is 0.10. That is the
difference in net benefit. The old policy reviews one AF recording and nothing
else, so its net benefit is one fifth, 0.20. The new policy reviews two with AF
and two without, two fifths less two fifths of 0.25, which is 0.30.

To bootstrap that paired difference, draw recording indices with replacement
and use the same draw for both models and the labels. Each resample's mean is
one possible cohort difference. The percentile interval describes sampling
uncertainty for independent recordings. It leaves out uncertainty from model
fitting, from choosing the prices and from moving to another service. Several
recordings from one patient would need the patient as the sampling unit.

### Problem D: Make

Implement `paired_bootstrap`. Draw every resample's indices in one call,
`indices = rng.integers(0, n, size=(resamples, n))`, average the contributions
in each row, and return a dictionary with the `estimate` and the 95%
percentile interval as `low` and `high`, from
`np.quantile(means, [0.025, 0.975])`. Keep the seeded generator. The checker
also accepts other valid ways of drawing the indices.

In [ ]:
def paired_bootstrap(old, new, y, preference, resamples=1000, seed=20261106):
    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=bool)
    change = (np.asarray(new) >= preference).astype(float) - (np.asarray(old) >= preference)
    contribution = change * np.where(y, 1., -preference / (1 - preference))
    n = len(y)
    indices = rng.integers(0, n, size=(resamples, n))
    means = contribution[indices].mean(axis=1)
    low, high = np.quantile(means, [0.025, 0.975])
    return {"estimate": contribution.mean(), "low": low, "high": high}
_ = check.run("cmp_bootstrap", paired_bootstrap)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# paired_bootstrap = check.use_reference("paired_bootstrap")

In [ ]:
if check.ready(paired_bootstrap=paired_bootstrap):
    print("New minus old net benefit:", paired_bootstrap(old, new, y, cut))

On development the difference in net benefit is -0.018 per recording, with
a 95% percentile interval from -0.024 to -0.012. The whole interval is below zero at
this preference. It is conditional on these two fixed policies.

## Explain the changed decisions

### Problem E: Make

Write `changed_recordings(ids, old, new, y, preference)`. Return a dictionary
of arrays named `recording`, `direction` (`"up"` or `"down"`) and
`contribution`, for the
recordings whose review decision changes only, in input order. The
contribution is new minus old value before dividing by the cohort size. Then
inspect one changed AF recording and one changed recording without AF: which
score crossed the cut, and which consequence changed? Scores and labels
support that account; they cannot show a waveform mechanism.

In [ ]:
def changed_recordings(ids, old, new, y, preference):
    a, b = np.asarray(old) >= preference, np.asarray(new) >= preference
    changed = a != b
    contribution = (b.astype(float) - a) * np.where(y, 1., -preference / (1 - preference))
    return {"recording": np.asarray(ids)[changed],
            "direction": np.where(b[changed], "up", "down"),
            "contribution": contribution[changed]}
_ = check.run("cmp_changed", changed_recordings)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# changed_recordings = check.use_reference("changed_recordings")

In [ ]:
if check.ready(changed_recordings=changed_recordings):
    audit = changed_recordings(cohort["recording"], old, new, y, cut)
    lookup = {recording: i for i, recording in enumerate(cohort["recording"])}
    examples = []
    for af in (True, False):
        selected = [rid for rid in audit["recording"] if y[lookup[rid]] == af]
        for rid in selected[:2]:
            i = lookup[rid]
            examples.append({"recording": rid, "reference": cohort["reference_label"][i],
                             "forest": round(float(old[i]), 3), "raw ResNet": round(float(new[i]), 3),
                             "old review": bool(old[i] >= cut), "new review": bool(new[i] >= cut)})
    display(view.table(list(examples[0]), examples, "Which recording changes action, and why?"))
    print("Sum of contributions / all recordings:", audit["contribution"].sum() / len(y))

### Problem F: Fix the bug

A draft report chooses the model with the code below. Fix `recommend` in place
so that it chooses by net benefit at the stated preference and returns
`"net_benefit"` as its basis. Then write in `explanation` one sentence that
separates the description (NRI) from the judgement (net benefit). The checker
tests the function, not your sentence.

In [ ]:
def recommend(old, new, y, preference):
    """Which model the report recommends, and on what basis."""
    gain = (comparison.policy_net_benefit(new, y, preference)
            - comparison.policy_net_benefit(old, y, preference))
    winner = "resnet" if gain > 0 else "random_forest"
    return {"winner": winner, "basis": "net_benefit", "preference": preference}


explanation = """
NRI describes how decisions move within each outcome group; net benefit at the
stated preference says whether those moves are worth their price, and here they
are not.
"""
_ = check.run("cmp_choice", recommend)

## Before you finish

- Account for the two NRI denominators and the neutral treatment of ties.
- Explain how a positive NRI can sit beside a negative difference in net benefit.
- State the preference, the difference, its interval and the sampling unit.
- Explain one changed decision using its ID, scores and reference label.

**Discuss:** If a service cared much more about misses, which calculation
would you redo before recommending a model?